> **Melodyne Labs restoration brief**
>
> A short synthesizer phrase survived a bad preset transfer, but its bass, mid, treble, and drive controls did not. Restore the hidden preset by making the renderer differentiable, measuring each raw control's responsibility, and letting an optimizer act on those gradients.
>
> The note phrase is newly authored here and never loaded from a recording. This lab complements the SmartVal XOR/manual-backprop notebook: SmartVal traces a neural network by hand; here the same chain rule travels through audio processing.

# Melodyne Backprop Lab: Restore the Synth

| Stage | Question | Evidence |
| --- | --- | --- |
| 1 | What signal are we restoring? | A deterministic 1.12-second, eight-note phrase generated here. |
| 2 | What is the forward pass? | Raw variables become bounded tone knobs, FFT equalization, and smooth drive. |
| 3 | Who is responsible for error? | `tf.GradientTape` assigns one gradient to each raw control. |
| 4 | Can we trust the gradients? | Central finite differences check all four independently. |
| 5 | Can an optimizer restore the preset? | CPU Adam records loss and knob histories; assertions guard recovery. |

The experiment is intentionally small enough for a laptop CPU. No audio files, pretrained models, or embedded outputs are required.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import Audio, display

SEED = 23
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.get_logger().setLevel("ERROR")
print(f"TensorFlow {tf.__version__} | NumPy {np.__version__}")
print("Optimization device: CPU (selected explicitly below)")

## 1. Author a Dry Phrase

The source is an original pattern written for this notebook: A2, C#3, E3, D3, G3, E3, B2, A2. Each note lasts 140 ms. A gently decaying bank of eighteen harmonics gives bass, mid, and treble controls real signal energy to adjust. A short cosine fade prevents clicks.

This is the **dry signal**: musical content before any recoverable controls are applied. The broad spectrum matters experimentally: a knob cannot be recovered from audio if the source contains no sound in the frequencies that knob controls.

In [ ]:
SAMPLE_RATE = 8_000
NOTE_SECONDS = 0.14
NOTE_SAMPLES = int(SAMPLE_RATE * NOTE_SECONDS)
PHRASE_HZ = np.array([110.00, 138.59, 164.81, 146.83, 196.00, 164.81, 123.47, 110.00], dtype=np.float32)

def synth_note(frequency_hz, note_index):
    time = np.arange(NOTE_SAMPLES, dtype=np.float32) / SAMPLE_RATE
    tone = np.zeros_like(time)
    for harmonic in range(1, 19):
        harmonic_phase = 0.17 * note_index + 0.11 * harmonic
        tone += np.sin(2.0 * np.pi * harmonic * frequency_hz * time + harmonic_phase) / harmonic
    fade_samples = int(0.018 * SAMPLE_RATE)
    fade = 0.5 - 0.5 * np.cos(np.linspace(0.0, np.pi, fade_samples, dtype=np.float32))
    envelope = np.ones(NOTE_SAMPLES, dtype=np.float32)
    envelope[:fade_samples] = fade
    envelope[-fade_samples:] = fade[::-1]
    return tone * envelope

dry_np = np.concatenate([synth_note(hz, index) for index, hz in enumerate(PHRASE_HZ)])
dry_np = (0.45 * dry_np / np.max(np.abs(dry_np))).astype(np.float32)
dry = tf.constant(dry_np)
time_axis = np.arange(dry_np.size) / SAMPLE_RATE
print(f"Phrase: {dry_np.size / SAMPLE_RATE:.2f} seconds, {dry_np.size:,} samples at {SAMPLE_RATE:,} Hz")
print("Harmonics span the bass, mid, and treble masks so every tone control is observable.")
display(Audio(dry_np, rate=SAMPLE_RATE, normalize=False))
assert dry_np.size == len(PHRASE_HZ) * NOTE_SAMPLES
assert np.max(np.abs(dry_np)) <= 0.451

## 2. Build the Differentiable Forward Pass

The renderer is a small computation graph:

$$r \longrightarrow k(r) \longrightarrow H(f;k) \longrightarrow x_{EQ}(t) \longrightarrow y(t)$$

1. Four unconstrained raw values pass through a sigmoid and become bounded, human-readable knobs.
2. Three smooth frequency masks blend bass, mid, and treble gains into $H(f;k)$.
3. `rfft` applies that response; `irfft` returns to the time domain.
4. A normalized `tanh` waveshaper applies drive without discontinuous clipping.

Why optimize raw values instead of the knobs directly? An optimizer may request any real-valued step. The sigmoid acts like a smooth linkage: raw values may move freely, while decoded controls always remain inside their permitted positive ranges. This prevents an update from producing an invalid control and keeps the entire mapping differentiable.

Every operation has a TensorFlow gradient, so an audio mismatch can send a useful correction all the way back to each raw control.

In [ ]:
CONTROL_NAMES = ("bass", "mid", "treble", "drive")
CONTROL_LOW = tf.constant([0.35, 0.35, 0.35, 0.60], dtype=tf.float32)
CONTROL_HIGH = tf.constant([1.65, 1.65, 1.65, 2.40], dtype=tf.float32)

def decode_controls(raw_controls):
    return CONTROL_LOW + (CONTROL_HIGH - CONTROL_LOW) * tf.sigmoid(raw_controls)

def encode_controls(controls):
    position = (tf.convert_to_tensor(controls, tf.float32) - CONTROL_LOW) / (CONTROL_HIGH - CONTROL_LOW)
    return tf.math.log(position / (1.0 - position))

frequency_hz = tf.cast(tf.range(dry_np.size // 2 + 1), tf.float32) * SAMPLE_RATE / dry_np.size
log_frequency = tf.math.log(frequency_hz + 45.0) / tf.math.log(2.0)
center_hz = tf.constant([120.0, 520.0, 2_200.0], dtype=tf.float32)
center_log = tf.math.log(center_hz + 45.0) / tf.math.log(2.0)
band_width = tf.constant([1.05, 0.95, 1.15], dtype=tf.float32)
band_logits = -0.5 * tf.square((log_frequency[:, None] - center_log[None, :]) / band_width[None, :])
band_masks = tf.nn.softmax(band_logits, axis=1)

def render(raw_controls):
    bass, mid, treble, drive = tf.unstack(decode_controls(raw_controls))
    response = tf.linalg.matvec(band_masks, tf.stack([bass, mid, treble]))
    equalized = tf.signal.irfft(
        tf.signal.rfft(dry) * tf.cast(response, tf.complex64),
        fft_length=[dry_np.size],
    )
    return tf.math.tanh(drive * equalized) / tf.math.tanh(drive)

assert band_masks.shape == (dry_np.size // 2 + 1, 3)
np.testing.assert_allclose(tf.reduce_sum(band_masks, axis=1).numpy(), 1.0, atol=1e-6)

### Hide a Preset, Then Damage It

The target controls are the secret preset. The damaged preset starts far away in all four dimensions: thin bass, excessive mids, muted treble, and too much drive.

Only target **audio** enters the objective; target knob values remain solely for evaluation. That separation is the point of the experiment: in a real restoration task you can hear the desired result, but you do not have the lost knob settings. If knob differences entered the loss, optimization could move directly toward the answer and would no longer prove that error can travel through the renderer from sound back to controls.

In [ ]:
target_controls = tf.constant([1.30, 0.72, 1.18, 1.55], dtype=tf.float32)
damaged_controls = tf.constant([0.45, 1.50, 0.52, 2.25], dtype=tf.float32)
target_raw = encode_controls(target_controls)
damaged_raw = encode_controls(damaged_controls)
target_audio = tf.stop_gradient(render(target_raw))
damaged_audio = tf.stop_gradient(render(damaged_raw))

print("Secret target:  " + ", ".join(f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, target_controls.numpy())))
print("Damaged start: " + ", ".join(f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, damaged_controls.numpy())))
print("Target reference:")
display(Audio(target_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
print("Damaged starting point:")
display(Audio(damaged_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
assert float(tf.reduce_max(tf.abs(target_audio))) < 1.0
assert float(tf.reduce_max(tf.abs(damaged_audio))) < 1.0

## 3. Loss, Backpropagation, and the Update

Waveform loss rewards sample agreement; log-spectrum loss rewards frequency balance. Neither view alone fully describes whether the restored sound matches:

$$L(r)=\operatorname{MSE}(y(r),y^*)+0.05\,\operatorname{MSE}(\log(1+20|Y(r)|),\log(1+20|Y^*|))$$

Read the two terms as one objective: match the target's time-domain waveform while also matching how its energy is distributed across frequencies. The factor `0.05` keeps the spectrum term useful without letting its numerical scale dominate.

- **Forward pass:** render with current controls and compute scalar loss.
- **Backpropagation:** apply the chain rule backward. Each $\partial L/\partial r_i$ reports how a tiny increase in one raw control would change loss; it measures direction and sensitivity but changes nothing.
- **Optimizer update:** consume those gradients. Plain gradient descent uses $r \leftarrow r-\eta\nabla_rL$; Adam adapts the step per control.

A positive gradient says increasing that raw control would raise loss, so subtraction moves it down. A negative gradient says increasing it would lower loss, so subtraction moves it up. **Backpropagation computes this guidance; gradient descent or Adam acts on it.** The target knobs do not appear in the loss.

In [ ]:
def log_magnitude_spectrum(audio):
    spectrum = tf.signal.rfft(audio)
    power = tf.square(tf.math.real(spectrum)) + tf.square(tf.math.imag(spectrum))
    return tf.math.log1p(20.0 * tf.sqrt(power + 1e-12))

target_log_spectrum = log_magnitude_spectrum(target_audio)

def restoration_loss(raw_controls):
    estimate = render(raw_controls)
    waveform_loss = tf.reduce_mean(tf.square(estimate - target_audio))
    spectrum_loss = tf.reduce_mean(tf.square(log_magnitude_spectrum(estimate) - target_log_spectrum))
    return waveform_loss + 0.05 * spectrum_loss

probe_raw = tf.Variable(damaged_raw)
with tf.GradientTape() as tape:
    probe_loss = restoration_loss(probe_raw)
autodiff_gradient = tape.gradient(probe_loss, probe_raw)

print(f"Forward-pass loss: {float(probe_loss):.6f}")
print("Backprop responsibility at the damaged preset:")
for name, gradient in zip(CONTROL_NAMES, autodiff_gradient.numpy()):
    direction = "decrease raw control" if gradient > 0 else "increase raw control"
    print(f"  dL/d{name:6s} = {gradient:+.7f} -> {direction}")
assert float(restoration_loss(target_raw)) < 1e-10
assert np.all(np.isfinite(autodiff_gradient.numpy()))

## 4. Audit All Four Gradients

Central finite differences provide a framework-independent local check:

$$\frac{\partial L}{\partial r_i}\approx\frac{L(r_i+\epsilon)-L(r_i-\epsilon)}{2\epsilon}$$

Four controls require eight extra forward passes. That is useful for diagnosis, but too costly and noisy to replace backpropagation during training.

In [ ]:
epsilon = tf.constant(1e-3, dtype=tf.float32)
finite_difference_gradient = []
for index in range(len(CONTROL_NAMES)):
    offset = tf.one_hot(index, len(CONTROL_NAMES), dtype=tf.float32) * epsilon
    loss_plus = restoration_loss(probe_raw + offset)
    loss_minus = restoration_loss(probe_raw - offset)
    finite_difference_gradient.append((loss_plus - loss_minus) / (2.0 * epsilon))
finite_difference_gradient = tf.stack(finite_difference_gradient)

print("control   GradientTape    finite diff    absolute error")
for name, automatic, numerical in zip(CONTROL_NAMES, autodiff_gradient.numpy(), finite_difference_gradient.numpy()):
    print(f"{name:7s}  {automatic:+.8f}   {numerical:+.8f}   {abs(automatic - numerical):.2e}")
np.testing.assert_allclose(autodiff_gradient.numpy(), finite_difference_gradient.numpy(), rtol=3e-2, atol=2e-5)
print("All four raw-control gradients pass the central-difference check.")

## 5. Restore the Preset and Listen to It Learn

Starting from the damaged raw values, repeat forward pass, loss, backpropagation, and optimizer update. Only four raw variables are trainable. Every pre-update loss and decoded knob position, plus the final state, is recorded for inspection.

The phrase is the entire training example, so this loop has no mini-batches of separate songs. To make progress audible without creating 220 audio players, we define one **listening batch** as 25 full-phrase optimizer updates. The next cell plays:

1. the fixed target reference;
2. the damaged starting sound at update 0;
3. the learned sound after each 25-update batch;
4. the final sound after update 220.

Each checkpoint is rendered only after its update has completed. Playback therefore demonstrates the optimizer's actual parameter state at that point; it does not alter gradients or training.

In [ ]:
raw_controls = tf.Variable(damaged_raw, name="raw_synth_controls")
optimizer = tf.keras.optimizers.Adam(learning_rate=0.06)
STEPS = 220
LISTEN_EVERY = 25
loss_history = []
knob_history = []
audio_checkpoints = []


def play_training_checkpoint(update_count):
    """Capture and display the synth state after a completed optimizer update."""
    checkpoint_loss = float(restoration_loss(raw_controls))
    checkpoint_controls = decode_controls(raw_controls).numpy().copy()
    checkpoint_audio = tf.stop_gradient(render(raw_controls)).numpy().copy()
    audio_checkpoints.append({
        "update": update_count,
        "loss": checkpoint_loss,
        "controls": checkpoint_controls,
        "audio": checkpoint_audio,
    })
    control_text = ", ".join(
        f"{name}={value:.2f}" for name, value in zip(CONTROL_NAMES, checkpoint_controls)
    )
    label = "damaged start" if update_count == 0 else f"after update {update_count}"
    print(f"\nListen — {label} | loss={checkpoint_loss:.6f} | {control_text}")
    display(Audio(checkpoint_audio, rate=SAMPLE_RATE, normalize=False))


print("Target reference — compare every checkpoint with this sound:")
display(Audio(target_audio.numpy(), rate=SAMPLE_RATE, normalize=False))
play_training_checkpoint(0)

with tf.device("/CPU:0"):
    for step in range(STEPS):
        with tf.GradientTape() as tape:
            loss = restoration_loss(raw_controls)
        gradient = tape.gradient(loss, raw_controls)
        loss_history.append(float(loss))
        knob_history.append(decode_controls(raw_controls).numpy().copy())
        optimizer.apply_gradients([(gradient, raw_controls)])

        completed_updates = step + 1
        if completed_updates % LISTEN_EVERY == 0 or completed_updates == STEPS:
            play_training_checkpoint(completed_updates)

loss_history.append(float(restoration_loss(raw_controls)))
knob_history.append(decode_controls(raw_controls).numpy().copy())
loss_history = np.asarray(loss_history)
knob_history = np.asarray(knob_history)
restored_controls = decode_controls(raw_controls).numpy()
restored_audio = tf.stop_gradient(render(raw_controls))

expected_checkpoints = [0, *range(LISTEN_EVERY, STEPS, LISTEN_EVERY), STEPS]
assert [checkpoint["update"] for checkpoint in audio_checkpoints] == expected_checkpoints
print(f"\nCompleted {STEPS} updates on /CPU:0")
print(f"Played {len(audio_checkpoints)} learning checkpoints: {expected_checkpoints}")
print(f"Loss: {loss_history[0]:.6f} -> {loss_history[-1]:.8f}")
for name, start, restored, target in zip(CONTROL_NAMES, damaged_controls.numpy(), restored_controls, target_controls.numpy()):
    print(f"  {name:7s}: {start:.3f} -> {restored:.3f} (target {target:.3f})")

### Recovery Gates

A lower training objective is necessary, but it is not the same as proving full recovery. Because the objective combines waveform and spectrum terms into one scalar, improvement in one term could hide a weaker result in the other. It also cannot by itself show that the optimizer found controls close to the known hidden preset.

The checks therefore inspect the result from independent views: objective reduction, time-domain waveform correlation, frequency-domain spectrum error, and physical-control error. Knob error is available here because this is a synthetic experiment with known ground truth; a real lost-preset restoration would rely on observable audio checks and listening tests instead.

In [ ]:
def log_spectrum_distance(audio_a, audio_b):
    spectrum_a = np.log1p(20.0 * np.abs(np.fft.rfft(audio_a)))
    spectrum_b = np.log1p(20.0 * np.abs(np.fft.rfft(audio_b)))
    return float(np.mean((spectrum_a - spectrum_b) ** 2))

target_np = target_audio.numpy()
damaged_np = damaged_audio.numpy()
restored_np = restored_audio.numpy()
waveform_correlation = float(np.corrcoef(target_np, restored_np)[0, 1])
damaged_spectrum_error = log_spectrum_distance(damaged_np, target_np)
restored_spectrum_error = log_spectrum_distance(restored_np, target_np)
knob_mae = float(np.mean(np.abs(restored_controls - target_controls.numpy())))
loss_reduction = 1.0 - loss_history[-1] / loss_history[0]
spectrum_reduction = 1.0 - restored_spectrum_error / damaged_spectrum_error

print(f"Loss reduction:       {loss_reduction:.2%}")
print(f"Waveform correlation: {waveform_correlation:.6f}")
print(f"Spectrum reduction:   {spectrum_reduction:.2%}")
print(f"Mean knob error:      {knob_mae:.4f}")
assert loss_history[-1] < 0.02 * loss_history[0], "Loss did not fall by at least 98%."
assert waveform_correlation > 0.995, "Restored waveform correlation is too low."
assert restored_spectrum_error < 0.05 * damaged_spectrum_error, "Spectrum did not recover enough."
assert knob_mae < 0.08, "Recovered controls are not close enough to the hidden preset."
print("Recovery gates passed.")

## 6. Inspect and Hear the Evidence

Waveforms reveal timing and sample shape; spectra reveal tonal balance and generated harmonics. The histories show whether loss fell smoothly and whether each decoded knob approached its hidden target. Dotted trajectory lines mark target controls.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
view = time_axis <= 0.42
axes[0, 0].plot(time_axis[view], target_np[view], label="target", color="#176B87", lw=1.8)
axes[0, 0].plot(time_axis[view], damaged_np[view], label="damaged", color="#C8553D", alpha=0.70)
axes[0, 0].plot(time_axis[view], restored_np[view], label="restored", color="#2E8B57", lw=1.1, ls="--")
axes[0, 0].set(title="Waveforms: first 420 ms", xlabel="Time (s)", ylabel="Amplitude")
axes[0, 0].legend()

spectrum_frequency = np.fft.rfftfreq(target_np.size, 1.0 / SAMPLE_RATE)
for audio, label, color in [(target_np, "target", "#176B87"), (damaged_np, "damaged", "#C8553D"), (restored_np, "restored", "#2E8B57")]:
    magnitude_db = 20.0 * np.log10(np.abs(np.fft.rfft(audio)) + 1e-5)
    axes[0, 1].plot(spectrum_frequency, magnitude_db, label=label, color=color, alpha=0.88)
axes[0, 1].set(xlim=(60, 2_600), ylim=(-65, 35), title="Magnitude spectra", xlabel="Frequency (Hz)", ylabel="Magnitude (dB)")
axes[0, 1].legend()

axes[1, 0].semilogy(loss_history, color="#4C3A69", lw=2)
axes[1, 0].set(title="Restoration objective", xlabel="Optimizer update", ylabel="Loss (log scale)")

knob_colors = ["#176B87", "#D17B0F", "#2E8B57", "#8D5A97"]
for index, (name, color) in enumerate(zip(CONTROL_NAMES, knob_colors)):
    axes[1, 1].plot(knob_history[:, index], label=name, color=color, lw=1.8)
    axes[1, 1].axhline(target_controls.numpy()[index], color=color, ls=":", alpha=0.75)
axes[1, 1].set(title="Decoded knob trajectories", xlabel="Optimizer update", ylabel="Knob value")
axes[1, 1].legend(ncol=2)
for axis in axes.flat:
    axis.grid(alpha=0.22)
plt.show()

print("Restored result:")
display(Audio(restored_np, rate=SAMPLE_RATE, normalize=False))

## What the Restoration Demonstrated

Target, damaged, and restored clips contain identical notes. Their difference lives entirely in the differentiable signal path. Early gradients identify how each raw control should move; Adam turns those responsibilities into updates.

The loop is the same one used by a neural network: parameters produce output, loss measures error, backpropagation computes responsibility, and an optimizer updates parameters. Only the model is unusual: a compact synthesizer instead of dense layers.

---

## Bridge: From Rendering a Phrase to Generating One

This lab assumed note events were known and learned **how they should sound**. The [RNN/LSTM sequence-modeling notebook](../05-rnn-sequence-modeling/rnn-sequence-modeling.ipynb) asks the complementary question: given earlier symbols, **what should come next?** It derives recurrent state, BPTT, vanishing gradients, and LSTM gates before the PyTorch bridge translates those contracts.

The later [Cinematic Piano Memory capstone](../07-pytorch-rnn-bridge/02-cinematic-piano-memory.ipynb) completes the arc with an original Dm-Bb-F-C motif. Transparent PyTorch RNN and LSTM cells generate actual note predictions, while a lightweight renderer makes checkpoint learning and long-horizon memory audible.

Connect the ideas by letting a sequence model emit note and duration tokens, translating those tokens into frequencies and timing, and passing that event list into a renderer. Tone controls can remain fixed, be predicted by another model, or be optimized against a reference timbre. Sequence modeling owns musical order; the renderer exposes timbre. Together they form a generate-then-render pipeline without replacing SmartVal's manual derivation of backpropagation.